# 03 · Parameter search (configuration B data)

Plan, worker commands, progress and ranking for one stage of the search defined in `configs/sweep.yml`. Training runs in terminals; computation lives in `src/model/sweep.py` and `src/viz.py`.

Pre-specified rules:
- Selection uses only label-free metrics on the MSB validation records excluding U2OS: reconstruction and θ control.
- Genotype results and the external sets (E1–E3) are not used for selection.
- All runs early-stop on the same MSB validation windows.

Stages: `bench` (throughput, optional) → `s1` (data mix × cut-off range × latent size, 12 runs) → `s2` (loss settings around the top 2 of s1, 8 runs) → `s3` (top 3 × 3 seeds, full length).

In [ ]:
# 1. Setup (no TensorFlow import)
import sys
from pathlib import Path
import pandas as pd

sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists() and (p / "configs").exists())
sys.path.insert(0, str(ROOT))
from src.model import sweep as sw
from src import viz

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 60)
%matplotlib inline
STAGE = "s1"          # bench / s1 / s2 / s3
print(ROOT)

## 2. Frozen plan and worker commands

`freeze_plan` writes the run list once to `results/sweep/<stage>/plan.json`; every worker reads it. `s2` and `s3` are derived from the ranking of earlier stages and are frozen after those stages finish. One worker per GPU; finished runs are skipped when a worker is restarted.

In [ ]:
plan = sw.freeze_plan(STAGE)
display(plan[["run_id", "overrides", "seed", "epochs", "patience"]])
_ = sw.commands(STAGE)

## 3. Progress
Status, epochs and an upper bound on the remaining time (assuming no early stopping).

In [ ]:
viz.eta_table(STAGE)

## 4. Learning curves

In [ ]:
viz.live_curves(STAGE, "val_OrigRecLoss");
viz.live_curves(STAGE, "val_FeatRecLoss");

## 5. Results and ranking

- `recon_r2`: median reconstruction R²
- `control_pass`: fraction of windows meeting all three θ-control conditions (amplitude ratio ≥ 1.5, monotonic in θ_L, shape correlation ≥ 0.95)
- `amp_ratio`, `thetaL_ctrl`, `shape_keep`: medians of the three conditions
- `rank_sum`: sum of ranks over the five criteria (lower is better)

In [ ]:
df = sw.collect(STAGE)
ranked = sw.rank(df)
cols = ["config_name", "rank_sum", "recon_r2", "control_pass", "amp_ratio", "thetaL_ctrl", "shape_keep",
        "z_active_dims", "epochs_run", "sec_per_epoch", "n_seeds"]
display(ranked[[c for c in cols if c in ranked]] if len(ranked) else "no finished runs yet")

In [ ]:
viz.factor_effects(df, name=STAGE + "_factor_effects");
viz.ranking_bars(ranked, name=STAGE + "_ranking");

## 6. Ranking across stages

Combined ranking of `s1`–`s3` and the YAML block of the selected configuration for `configs/circadian_vabam.yml` (`vabam: Models:`).

In [ ]:
all_df = sw.collect(["s1", "s2", "s3"])
all_ranked = sw.rank(all_df)
display(all_ranked[[c for c in cols + ["stages"] if c in all_ranked]].head(10) if len(all_ranked) else "no finished runs yet")
if len(all_ranked):
    _ = sw.best_yaml(all_ranked, k=1)

## Appendix · batch size / XLA throughput

```
python run.py sweep --stage bench --worker 0 --nworkers 1 --gpu 0
```

In [ ]:
try:
    display(sw.bench_table("bench"))
except FileNotFoundError:
    print("bench stage has not been run")